# Stage 7 — Interpretation: what the clusters mean, where they are, what is odd

**Data:** Porto taxi trajectories (UCI 339), real GPS — `synthetic: false`.
**Limitation, repeated on purpose:** these are **taxis, not buses**. No fixed stops, no
timetable, no schedule delay. "Dwell" means below walking pace, not boarding passengers.
Any statement about bus passengers or footfall would be unsupported by this data.

**Chosen clustering (Stage 6):** K-Means, k = 4, on PC1–PC5. TS-K-Means on 168-hour
profiles is kept as a *second, orthogonal* label set: a cell can be congested *and*
morning-peaked, and those are different questions.

**What this notebook produces**

| Output | Content |
|---|---|
| `outputs/cluster_profiles.csv` | one row per cluster: size, centroid, mean of every feature, name |
| `outputs/cluster_pois.csv` | POI counts and densities per cluster (source: OSM) |
| `outputs/cluster_hourly_profile.parquet` | cluster x 168-hour mean pings per cell |
| `outputs/cluster_rhythm_crosstab.csv` | K-Means cluster x weekly-rhythm cluster |
| `outputs/cell_annotations.parquet` | per-cell labels, names, anomaly scores and flags |
| `outputs/anomalies.csv` | flagged cells only, with the features that drove the flag |
| `outputs/day_anomalies.csv` | per-day counts, speed, z-scores, flags |
| `outputs/figures/stage7_*.png` | profile heatmap + rhythms, POI densities, anomaly map |
| `docs/PRIVACY_AND_LIMITATIONS.md` | written by this notebook, not by hand |

In [1]:
import json, pathlib, sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # notebooks/ runs one level below the repo root

import numpy as np
import pandas as pd

from src.io import PROJECT_ROOT          # same convention as notebooks 01-06
from src import interpret as I

PROC = PROJECT_ROOT / "data" / "processed"
OUT = PROJECT_ROOT / "outputs"; OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
DOCS = PROJECT_ROOT / "docs"; DOCS.mkdir(parents=True, exist_ok=True)

cells    = pd.read_parquet(PROC / "cells_features.parquet")
pca      = pd.read_parquet(PROC / "pca_scores.parquet")
counts   = pd.read_parquet(PROC / "hour_of_week_counts.parquet")   # index = h3_cell
profile  = pd.read_parquet(PROC / "hour_of_week_profile.parquet")
trips    = pd.read_parquet(PROC / "trip_features.parquet")
meta_in  = json.load(open(PROC / "features_meta.json"))

lab_k    = pd.read_parquet(PROC / "labels_kmeans.parquet").set_index("h3_cell")["kmeans"]
lab_ts   = pd.read_parquet(PROC / "labels_tskmeans.parquet").set_index("h3_cell")["tskmeans"]

print("cells:", len(cells), "| labelled by kmeans:", int(lab_k.notna().sum()),
      "| labelled by tskmeans:", int(lab_ts.notna().sum()))
print("data_source:", meta_in["data_source"], "| synthetic:", meta_in["synthetic"])

cells: 602 | labelled by kmeans: 602 | labelled by tskmeans: 602
data_source: porto_taxi_uci339 | synthetic: False


## 1. Cluster profiles — the numbers behind every name

For each cluster we take the **mean of every feature over its cells**. These are averages
of averages (a cell's speed is itself an average of its steps), so they describe *typical
behaviour of the cells in the cluster*, not any individual trip.

The POI join adds OpenStreetMap context, counted per H3 cell so that a large cluster does
not "own more schools" merely by covering more area.

In [2]:
bbox   = I.bbox_from_cells(cells)
pois   = I.fetch_pois(bbox)                    # cached in data/raw/osm/ after first run
poi_tab = I.poi_cell_counts(pois)

prof     = I.name_clusters(I.cluster_profiles(cells, lab_k),
                           I.cluster_poi_profile(cells, lab_k, poi_tab))
poi_prof = I.cluster_poi_profile(cells, lab_k, poi_tab)
hourly   = I.cluster_hourly_profile(counts, lab_k)

print("OSM POIs fetched:", len(pois), "| categories:", sorted(pois['category'].unique()))
print("analysed cells with >= 1 POI:", int(poi_tab.index.isin(cells['h3_cell']).sum()),
      "of", len(cells), "(the POI table spans the whole bbox, so it holds more cells than we analyse)")

show = ["cluster", "n_cells", "share_cells", "name", "rhythm_modifier", "poi_context",
        "peak_hour", "speed_mean", "speed_p95", "slow_share", "speed_cv",
        "dwell_s_per_visit", "n_visits", "congestion_index", "volume_index"]
print("\n=== PROFILE (means over cells) ===")
print(prof[show].round(3).to_string(index=False))

print("\n=== ONE-LINE READINGS (numbers are measured, not asserted) ===")
for _, r in prof.iterrows():
    print(f"C{int(r.cluster)}  {r.label}\n     {r.description}")

OSM POIs fetched: 22579 | categories: ['civic', 'education', 'health', 'hospitality', 'industry', 'market', 'retail_area', 'transport', 'worship']
analysed cells with >= 1 POI: 585 of 602 (the POI table spans the whole bbox, so it holds more cells than we analyse)

=== PROFILE (means over cells) ===
 cluster  n_cells  share_cells                       name rhythm_modifier                     poi_context  peak_hour  speed_mean  speed_p95  slow_share  speed_cv  dwell_s_per_visit  n_visits  congestion_index  volume_index
       0      121        0.201   Slow low-traffic pockets   weekend-heavy                                         18      34.274     65.056       0.236     0.662             57.014    26.802             0.519        -0.592
       1      164        0.272   Busy mid-speed corridors         all-day hospitality-dense (5.6x median)         17      34.009     72.788       0.288     0.755             21.570  2339.482             0.122         1.732
       2      130        0.216

### How the names were produced (so they can be argued with)

* **Behaviour axis.** `congestion_index` = mean z of (slow_share, speed_cv,
  dwell_s_per_visit) minus mean z of (speed_mean, speed_p95); `volume_index` = mean z of
  (n_visits, n_vehicles). Congestion is cut into three bands at ±1 index unit, so
  "severely congested" and "slow" are different words.
* **Rhythm modifier.** The largest of night / AM / PM / weekend shares *inside* the
  cluster, kept only if it exceeds the median cluster's share for that same window.
* **POI phrase.** For each category, this cluster's POI density ÷ the median cluster's
  density; the largest ratio wins if it is ≥ 2. `transport` (bus stops) is excluded from
  the phrase because bus-stop density tracks the road network and would tag every cluster
  the same — a measured lesson, not a preference.

In [3]:
print("=== POI density per cell by cluster ===")
cols = ["cluster"] + [f"{c}_per_cell" for c in I.POI_CATEGORIES] + ["poi_per_cell"]
print(poi_prof[cols].round(2).to_string(index=False))

print("\n=== share of the city's clusters ===")
print((prof.set_index("label")["share_cells"] * 100).round(1).to_string())

=== POI density per cell by cluster ===
 cluster  education_per_cell  market_per_cell  transport_per_cell  health_per_cell  hospitality_per_cell  civic_per_cell  worship_per_cell  retail_area_per_cell  industry_per_cell  poi_per_cell
       0                1.02             0.56                9.64             0.51                  2.51            0.48              0.58                  0.36               1.01         16.67
       1                3.07             1.74               17.46             2.21                 18.20            2.87              1.88                  1.06               2.14         50.63
       2                1.24             0.75               11.36             0.81                  3.95            1.03              0.82                  0.82               1.86         22.64
       3                0.30             0.20                5.80             0.19                  1.00            0.11              0.35                  0.38               0.98     

## 2. The orthogonal axis: weekly rhythms (TS-K-Means)

TS-K-Means clustered the **shape of the week**, not the average behaviour. A cluster of
cells can therefore be split by *when* it is used, which is exactly the "temporal regime"
question from the brief.

Naming uses three measured properties, because peak hour alone gave three of four clusters
the same name in my first attempt: weekend share (0.30 / 0.12 cuts), peak shape (one run
vs ≥ 2 weekday runs above 80 % of the weekday maximum) and activity level (> 3× or < 0.5×
the median cluster's total).

In [4]:
rhythm_prof = I.cluster_hourly_profile(counts, lab_ts)
rhythm_names = I.name_rhythms(rhythm_prof)
print(rhythm_names.to_string(index=False))

cross = (pd.crosstab(lab_k, lab_ts, rownames=["kmeans_cluster"], colnames=["rhythm_cluster"]))
print("\n=== K-Means cluster x rhythm cluster (cell counts) ===")
print(cross.to_string())
print("\nReading: a K-Means cluster spread across several rhythm clusters is a behaviour")
print("that happens at different times of week in different places.")

 rhythm_cluster                                                 rhythm_name  peak_hour  n_peaks  weekend_share activity_level  mean_total_activity
              0       all-week, morning twin-peak (09:00 peak, high-volume)          9        2          0.261    high-volume               6040.1
              1 weekend-weighted, midday twin-peak (14:00 peak, mid-volume)         14        2          0.344     mid-volume                 57.8
              2      all-week, morning single-peak (09:00 peak, mid-volume)          9        1          0.228     mid-volume                 61.6
              3    weekday-only, morning twin-peak (09:00 peak, mid-volume)          9        2          0.091     mid-volume                104.8

=== K-Means cluster x rhythm cluster (cell counts) ===
rhythm_cluster    0   1   2   3
kmeans_cluster                 
0                81  24  12   4
1               164   0   0   0
2                88  12  13  17
3               137  23  25   2

Reading: a K-Mea

In [5]:
OUT.mkdir(exist_ok=True)
prof.drop(columns=["geography"], errors="ignore").to_csv(OUT / "cluster_profiles.csv", index=False)
poi_prof.to_csv(OUT / "cluster_pois.csv", index=False)
hourly.to_parquet(OUT / "cluster_hourly_profile.parquet")
rhythm_names.to_csv(OUT / "cluster_rhythm_names.csv", index=False)
cross.to_csv(OUT / "cluster_rhythm_crosstab.csv")
print("wrote cluster_profiles.csv, cluster_pois.csv, cluster_hourly_profile.parquet,")
print("      cluster_rhythm_names.csv, cluster_rhythm_crosstab.csv")

wrote cluster_profiles.csv, cluster_pois.csv, cluster_hourly_profile.parquet,
      cluster_rhythm_names.csv, cluster_rhythm_crosstab.csv


## 3. Anomalies — three different questions

| Kind | Question | Score | Flag rule |
|---|---|---|---|
| **Behaviour** | Which cells do not look like the rest of their cluster? | $\lVert x_i-\mu_{k(i)}\rVert_2$ in PC1–PC5 | top 2 % **and** robust $z\ge2$ |
| **Rhythm** | Which cells are used at the wrong times for their cluster? | $\lVert p_i-\bar p_{k(i)}\rVert_2$ on row-normalised 168-h profiles | top 2 % **and** robust $z\ge2$ |
| **Day** | Which calendar days deviate? | quasi-Poisson $z$ on trip count; robust $z$ on median speed | $|z|\ge2.5$ with ≥ 30 sampled trips |

Two conditions per cell, because either alone misbehaves: a pure rank cut always flags
someone even when nothing is odd, and a pure $z$ cut flags half the city when the
distribution is heavy-tailed. Robust $z=(x-\text{median})/(1.4826\,\text{MAD})$ is used so
the outliers we are hunting cannot inflate the very scale used to judge them.

In [6]:
spa = I.spatial_anomalies(pca, lab_k)
tmp = I.temporal_anomalies(profile, lab_k)
drv = I.driver_features(cells, lab_k)

ann = (spa.merge(tmp.drop(columns="cluster"), on="h3_cell", how="outer")
          .merge(drv, on="h3_cell", how="left"))
ann["rhythm_cluster"] = ann["h3_cell"].map(lab_ts)
ann["cluster_name"] = ann["cluster"].map(prof.set_index("cluster")["label"])
ann["anomaly_type"] = np.select(
    [ann.is_anomaly_spatial & ann.is_anomaly_temporal,
     ann.is_anomaly_spatial, ann.is_anomaly_temporal],
    ["both", "behaviour", "rhythm"], default="none")
ann["is_anomaly"] = ann.anomaly_type != "none"

print("flagged behaviour:", int(ann.is_anomaly_spatial.sum()),
      "| rhythm:", int(ann.is_anomaly_temporal.sum()),
      "| both:", int((ann.is_anomaly_spatial & ann.is_anomaly_temporal).sum()),
      "| total cells:", len(ann))
print(ann.anomaly_type.value_counts().to_string())

cols = ["h3_cell", "cluster", "anomaly_type", "distance_to_centroid",
        "profile_distance", "drivers"]
print("\n=== flagged cells (sorted by behaviour distance) ===")
print(ann.loc[ann.is_anomaly].sort_values("distance_to_centroid", ascending=False)[cols]
        .head(12).to_string(index=False))

flagged behaviour: 13 | rhythm: 7 | both: 3 | total cells: 602
anomaly_type
none         585
behaviour     10
rhythm         4
both           3

=== flagged cells (sorted by behaviour distance) ===
        h3_cell  cluster anomaly_type  distance_to_centroid  profile_distance                                                       drivers
8839220087fffff        2    behaviour                6.6408           0.54569         speed_cv +5.0sd; slow_share +3.0sd; speed_mean -2.2sd
8839220de1fffff        3         both                6.4014           0.58574   pm_peak_share +4.4sd; weekend_share -1.9sd; speed_cv +1.3sd
8839220011fffff        2         both                6.2085           0.65767   am_peak_share +3.5sd; slow_share +1.5sd; night_share -1.3sd
8839220521fffff        2    behaviour                5.8428           0.38177      pm_peak_share +3.7sd; slow_share +1.9sd; speed_cv +1.6sd
8839220523fffff        2    behaviour                5.4923           0.47419      pm_peak_share +3.3s

## 4. Day-level anomalies, with the sample-size problem handled

This dataset is a systematic sample (~1/28) of all Porto taxi trips, so daily counts are
*estimates*. Two corrections matter and both are computed, not assumed:

1. **Compare like with like** — each day is compared with the median for the same weekday.
2. **Over-dispersion** — Poisson assumes variance = mean, but demand varies more than that.
   Measured $\varphi \approx 3.2$ here, so a raw Poisson $z$ flags ~50 days out of 359.
   Scaling by $\sqrt{\varphi}$ reduces this to the handful that are genuinely unusual.

Days with fewer than 30 sampled trips are marked "not assessable" rather than flagged.

In [7]:
day = I.day_anomalies(trips, z_min=2.5, min_trips=30)
print(f"days covered: {len(day)} | dispersion phi = {day.dispersion_phi.iloc[0]}")
print(f"volume flags: {int(day.is_anomaly_volume.sum())} | speed flags: "
      f"{int(day.is_anomaly_speed.sum())} | not assessable: {int((day.note != '').sum())}")

flag_days = day.loc[day.is_anomaly_volume | day.is_anomaly_speed,
                    ["date", "n_trips", "poisson_z", "volume_z", "median_speed",
                     "median_duration", "speed_robust_z", "speed_direction",
                     "is_anomaly_volume", "is_anomaly_speed"]]
print("\n=== flagged days ===")
print(flag_days.to_string(index=False))
print("\nCheck the list against public holidays: 15 Aug, 25 Dec, 1 Jan should appear.")

days covered: 365 | dispersion phi = 3.207
volume flags: 8 | speed flags: 4 | not assessable: 6

=== flagged days ===
      date  n_trips  poisson_z  volume_z  median_speed  median_duration  speed_robust_z speed_direction  is_anomaly_volume  is_anomaly_speed
2013-08-15      107     -4.675    -2.610         26.96            525.0           1.469                               True             False
2013-08-18      143     -1.193    -0.666         30.31            540.0           2.741          faster              False              True
2013-08-23      121     -4.766    -2.661         25.39            525.0           0.875                               True             False
2013-11-26      227      6.185     3.454         21.17            630.0          -0.725                               True             False
2013-12-25       87     -5.176    -2.890         30.20            540.0           2.696          faster               True              True
2014-01-01      228      6.317     3

## 5. Figures

1. **Profile heatmap** — cluster means as z-scores, so "unusual" is visible at a glance.
2. **Weekly rhythms** — mean pings per cell per hour, one line per cluster.
3. **POI density** — the context behind the names.
4. **Anomaly map** — flagged cells over all cells, plus the score distributions.

In [8]:
I.plot_profiles(prof, hourly, FIG / "stage7_profiles.png")
I.plot_poi(poi_prof, prof, FIG / "stage7_poi.png")
I.plot_anomalies(cells, ann, FIG / "stage7_anomalies.png")
print("wrote:")
for p in sorted(FIG.glob("stage7_*.png")):
    print(" ", p, f"{p.stat().st_size/1024:.0f} KB")

wrote:
  /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage7_anomalies.png 85 KB
  /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage7_poi.png 58 KB
  /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage7_profiles.png 189 KB


In [9]:
ann_out = ann[["h3_cell", "cluster", "cluster_name", "rhythm_cluster", "anomaly_type",
               "is_anomaly", "distance_to_centroid", "anomaly_z_spatial",
               "profile_distance", "anomaly_z_temporal", "drivers"]].copy()
ann_out = ann_out.merge(cells[["h3_cell", "lat_centroid", "lon_centroid", "peak_hour",
                               "speed_mean", "slow_share", "n_visits"]], on="h3_cell", how="left")
ann_out.to_parquet(OUT / "cell_annotations.parquet", index=False)

# lat/lon live in `cells`, not in the annotation frame - join before writing the CSV
ann_geo = ann.merge(cells[["h3_cell", "lat_centroid", "lon_centroid"]], on="h3_cell", how="left")
ann_geo.loc[ann_geo.is_anomaly, cols + ["anomaly_z_spatial", "anomaly_z_temporal",
                                       "lat_centroid", "lon_centroid"]].sort_values(
    "anomaly_z_spatial", ascending=False).to_csv(OUT / "anomalies.csv", index=False)
day.to_csv(OUT / "day_anomalies.csv", index=False)
print("wrote cell_annotations.parquet ({:,.0f} cells), anomalies.csv ({:,.0f} flagged), day_anomalies.csv ({:,.0f} days)"
      .format(len(ann_out), int(ann.is_anomaly.sum()), len(day)))

wrote cell_annotations.parquet (602 cells), anomalies.csv (17 flagged), day_anomalies.csv (365 days)


## 6. Limitations and privacy

Written by the pipeline itself so the caveats cannot drift away from the results.

In [10]:
meta = I.interpret_meta(meta_in)
meta["limitation_text"] = (
    "Real GPS from Porto taxis (UCI 339), not city buses. There are no fixed stops, no "
    "timetable and no schedule delay to measure, so route-segment delay and stop profiles "
    "cannot be computed from this source. 'Dwell' is below-walking-pace time, which mixes "
    "kerbside waiting, parking and traffic. Cluster names describe taxi movement, not "
    "passenger demand."
)
doc_path = DOCS / "PRIVACY_AND_LIMITATIONS.md"
I.write_limitations(doc_path, meta)
json.dump(meta, open(OUT / "interpret_meta.json", "w"), indent=1)
print(f"wrote {doc_path} ({len(doc_path.read_text())} chars)")
print(doc_path.read_text()[:1200], "...")

wrote /home/ashira/Documents/Projects/Fleetsense/docs/PRIVACY_AND_LIMITATIONS.md (3949 chars)
# Limitations and privacy

_Generated by `src/interpret.py` (Stage 7). Dataset: `porto_taxi_uci339`,
synthetic: **False**._

## 1. What the data is not
Real GPS from Porto taxis (UCI 339), not city buses. There are no fixed stops, no timetable and no schedule delay to measure, so route-segment delay and stop profiles cannot be computed from this source. 'Dwell' is below-walking-pace time, which mixes kerbside waiting, parking and traffic. Cluster names describe taxi movement, not passenger demand.

* **No ground truth.** Nothing in this project knows what a "footfall hotspot" is. The
  clusters are a description of the data, not a validated classification. Every name
  ("morning-peaked congested core") is a hypothesis written next to its numbers.
* **Sampling, not census.** The trip table is a systematic sample of the source file, so
  absolute counts are not city totals. Relative comparisons 

## 7. What the analysis says (and what it does not)

Read the printed tables, not the names, when making a claim. Concretely, this notebook
supports statements of the form *"cells in cluster C2 have a mean speed of X km/h and an
AM-peak share of Y, and contain Z hospitality POIs per cell"* — every number is in
`outputs/cluster_profiles.csv` and `outputs/cluster_pois.csv`. It does **not** support
*"C2 is a nightlife district"*: POI density is context, not a cause, and taxi exposure
bias means a place with no taxis looks empty whether or not people are there.